# Multiplicative miscalibration of binary logistic models: supplementary information

This notebook (R kernel) contains analyses that support the paper but are not part of it.

## S1. Robustness of Section 5 to the train/test split

In the paper, the first 70% of each recording (speaker × story) is used for training and the last 30% for testing. Here we repeat the whole analysis of Section 5 for other splits:

* **Position of the test block.** The test set is a contiguous block of 30% of each recording, placed at offsets 0%, 10%, ..., 70% of the recording. Offset 70% is the split used in the paper.
* **Size of the training set.** The test set is the last part of each recording, and the training set is the first 50%, 60%, 70% or 80%.

For each split we refit everything on the training clips: the two-dimensional discriminant projection, the pairwise LDA and Firth classifiers, the full (majority-vote) models and the three partial models. We then report test accuracies and, for the Firth partial models, the accuracy after temperature scaling with temperatures chosen by blocked 10-fold cross-validation on the training clips.

The test blocks are contiguous in time, because neighbouring clips of a recording are strongly correlated; a random split by clip would let the test set leak into the training set.

Required R packages: none beyond base R. Runs in a few minutes.

In [ ]:
emb <- read.csv("data/ecapa_embeddings.csv")
emb$speaker <- factor(emb$speaker)
spk <- levels(emb$speaker)                      # "A" "M" "S"
X192 <- as.matrix(emb[, grep("^dim_", names(emb))])
rec  <- paste(emb$speaker, emb$story)           # one recording per speaker and story
## Position of each clip within its recording, from 0 (first clip) to 1 (last).
pos <- ave(emb$start_ms, rec, FUN = function(t) (rank(t) - 1) / (length(t) - 1))
cat(nrow(emb), "clips,", length(unique(rec)), "recordings\n")

### The pipeline of Section 5 as a function of the split

The code below follows the paper: a regularized discriminant projection to two dimensions (shrinkage 0.5), pairwise LDA and Firth logistic regression, majority voting for the full model and the closed-form partial model of Section 5. Temperatures for the two remaining Firth classifiers of a partial model maximize the likelihood of the coupled three-class model on out-of-fold log-odds.

In [ ]:
alpha3 <- 0.5
pairs3 <- combn(3, 2, simplify = FALSE)
drops  <- list(drop12 = c(i = 1, j = 2, k = 3),
               drop13 = c(i = 1, j = 3, k = 2),
               drop23 = c(i = 2, j = 3, k = 1))
drop_lab <- c(drop12 = "drop (A,M)", drop13 = "drop (A,S)", drop23 = "drop (M,S)")

proj_fit <- function(X, y) {
  nk <- as.integer(table(y)); mu <- t(sapply(spk, function(k) colMeans(X[y == k, ])))
  Sw <- Reduce(`+`, lapply(1:3, function(i) (nk[i] - 1) * cov(X[y == spk[i], ]))) / (nrow(X) - 3)
  Sw <- (1 - alpha3) * Sw + alpha3 * mean(diag(Sw)) * diag(ncol(X))
  e  <- eigen(Sw, symmetric = TRUE)
  Wh <- e$vectors %*% diag(1 / sqrt(pmax(e$values, 1e-12))) %*% t(e$vectors)
  Wh %*% eigen(cov(mu %*% Wh) * 2, symmetric = TRUE)$vectors[, 1:2]
}
firth_beta <- function(X, y, tol = 1e-8, max_iter = 200) {
  X <- cbind(1, X); beta <- rep(0, ncol(X))
  for (it in 1:max_iter) {
    p <- plogis(as.vector(X %*% beta)); W <- p * (1 - p)
    Ai <- solve(t(X) %*% (X * W))
    h <- W * rowSums((X %*% Ai) * X)
    step <- as.vector(Ai %*% (t(X) %*% (y - p + h * (0.5 - p))))
    beta <- beta + step
    if (max(abs(step)) < tol) break
  }
  beta
}
lda_beta <- function(X, y) {                    # log-odds of y = 1 over y = 0
  X0 <- X[y == 0, ]; X1 <- X[y == 1, ]; n0 <- nrow(X0); n1 <- nrow(X1)
  m0 <- colMeans(X0); m1 <- colMeans(X1)
  Si <- solve(((n0 - 1) * cov(X0) + (n1 - 1) * cov(X1)) / (n0 + n1 - 2))
  c(-0.5 * as.numeric(m1 %*% Si %*% m1 - m0 %*% Si %*% m0) + log(n1 / n0),
    as.numeric(Si %*% (m1 - m0)))
}
## Fit the pairwise classifiers of one type on projected training data Z, g
## (classes 1..3); returns a function giving the log-odds of class a over b.
fit_pairs <- function(Z, g, fitter) {
  B <- list()
  for (pr in pairs3) { s <- g %in% pr; B[[paste(pr, collapse = "_")]] <- fitter(Z[s, ], as.integer(g[s] == pr[2])) }
  function(a, b, Znew) {
    lo <- min(a, b); hi <- max(a, b)
    eta <- as.numeric(cbind(1, Znew) %*% B[[paste(lo, hi, sep = "_")]])
    if (a == hi) eta else -eta
  }
}
vote_full <- function(Z, lo) {
  v <- matrix(0, nrow(Z), 3)
  for (pr in pairs3) { w <- lo(pr[1], pr[2], Z) > 0
    v[w, pr[1]] <- v[w, pr[1]] + 1; v[!w, pr[2]] <- v[!w, pr[2]] + 1 }
  v
}
partial_scores <- function(Z, d, lo) {          # log-odds of i and j over the linkage class k
  cbind(lo(d[["i"]], d[["k"]], Z), lo(d[["j"]], d[["k"]], Z))
}
partial_L <- function(sc, d, u = c(0, 0)) {     # u = log temperatures
  L <- matrix(0, nrow(sc), 3)
  L[, d[["i"]]] <- sc[, 1] / exp(u[1]); L[, d[["j"]]] <- sc[, 2] / exp(u[2])
  L
}
coupled_nll <- function(u, sc, d, y) {
  L <- partial_L(sc, d, u); m <- apply(L, 1, max)
  -sum(L[cbind(seq_along(y), y)] - m - log(rowSums(exp(L - m))))
}
## Contiguous blocks in time within each training recording, rotated by 'shift'.
block_folds <- function(rec_tr, pos_tr, K, shift) {
  fd <- integer(length(rec_tr))
  for (r in unique(rec_tr)) {
    ix <- which(rec_tr == r); ix <- ix[order(pos_tr[ix])]; n <- length(ix)
    fd[ix] <- floor(((seq_len(n) - 1 + round(shift * n)) %% n) / n * K) + 1
  }
  fd
}
## Temperatures for the Firth partial model with pair d dropped, by blocked K-fold CV.
cv_temperatures <- function(Xtr, gtr, rec_tr, pos_tr, d, K = 10, shift = 0) {
  fd <- block_folds(rec_tr, pos_tr, K, shift); sc <- matrix(NA, length(gtr), 2)
  for (f in 1:K) {
    a <- fd != f; P <- proj_fit(Xtr[a, ], spk[gtr[a]])
    lo <- fit_pairs(Xtr[a, ] %*% P, gtr[a], firth_beta)
    sc[!a, ] <- partial_scores(Xtr[!a, ] %*% P, d, lo)
  }
  optim(c(0, 0), coupled_nll, sc = sc, d = d, y = gtr)$par
}

analyze_split <- function(is_test, n_shifts = 5) {
  tr <- !is_test
  Xtr <- X192[tr, ]; Xte <- X192[is_test, ]
  gtr <- as.integer(emb$speaker[tr]); gte <- as.integer(emb$speaker[is_test])
  P <- proj_fit(Xtr, emb$speaker[tr]); Ztr <- Xtr %*% P; Zte <- Xte %*% P
  lo <- list(LDA = fit_pairs(Ztr, gtr, lda_beta), Firth = fit_pairs(Ztr, gtr, firth_beta))
  acc <- function(L) mean(max.col(L, ties.method = "first") == gte)
  out <- list()
  for (m in names(lo)) out[[paste(m, "full")]] <- acc(vote_full(Zte, lo[[m]]))
  for (dn in names(drops)) for (m in names(lo))
    out[[paste(m, drop_lab[dn])]] <- acc(partial_L(partial_scores(Zte, drops[[dn]], lo[[m]]), drops[[dn]]))
  ## Firth partial models with CV temperatures (mean over n_shifts block placements).
  shifts <- (seq_len(n_shifts) - 1) / n_shifts
  for (dn in names(drops)) {
    d <- drops[[dn]]; sc_te <- partial_scores(Zte, d, lo$Firth)
    out[[paste("Firth", drop_lab[dn], "+ T")]] <- mean(sapply(shifts, function(s)
      acc(partial_L(sc_te, d, cv_temperatures(Xtr, gtr, rec[tr], pos[tr], d, 10, s)))))
  }
  ## Training Mahalanobis distances between the class means in the projected plane.
  for (pr in pairs3) {
    a <- gtr == pr[1]; b <- gtr == pr[2]
    S <- ((sum(a) - 1) * cov(Ztr[a, ]) + (sum(b) - 1) * cov(Ztr[b, ])) / (sum(a) + sum(b) - 2)
    dm <- colMeans(Ztr[a, ]) - colMeans(Ztr[b, ])
    out[[sprintf("d(%s,%s)", spk[pr[1]], spk[pr[2]])]] <- sqrt(as.numeric(t(dm) %*% solve(S) %*% dm))
  }
  unlist(out)
}

### Check: the split used in the paper

With the test set at the end of each recording, the code reproduces the accuracies of the paper (full models 95% and 94%; Firth partial model with (A,S) dropped 67%).

In [ ]:
paper_split <- emb$set == "test"
round(100 * analyze_split(paper_split)[1:8])

### Position of the 30% test block

In each recording, the test block has as many clips as in the paper (20 or 21 of 62–64). It starts at 0%, 10%, ..., 70% of the way from the first possible start to the last; the last position is exactly the split of the paper.

In [ ]:
offsets <- seq(0, 0.7, by = 0.1)
rk  <- ave(emb$start_ms, rec, FUN = rank) - 1                 # 0, 1, ..., n - 1 within a recording
n_r <- ave(emb$start_ms, rec, FUN = length)
m_r <- ave(as.numeric(paper_split), rec, FUN = sum)           # paper's test clips per recording
in_block <- function(o) { s <- round(o / 0.7 * (n_r - m_r)); rk >= s & rk < s + m_r }
stopifnot(all(in_block(0.7) == paper_split))
res_pos <- t(sapply(offsets, function(o) analyze_split(in_block(o))))
rownames(res_pos) <- sprintf("test block at %d%%", round(100 * offsets))
acc_cols <- grep("^(LDA|Firth)", colnames(res_pos))
round(100 * res_pos[, acc_cols])

In [ ]:

round(res_pos[, grep("^d\\(", colnames(res_pos))], 1)

### Size of the training set

The test set is the last part of each recording and the training set the first 50%, 60%, 70% or 80%. The test set is scaled from the paper's (70% training), so that 70% reproduces the paper.

In [ ]:
train_frac <- c(0.5, 0.6, 0.7, 0.8)
last_part <- function(f) rk >= n_r - round(m_r * (1 - f) / 0.3)
stopifnot(all(last_part(0.7) == paper_split))
res_size <- t(sapply(train_frac, function(f) analyze_split(last_part(f))))
rownames(res_size) <- sprintf("train %d%%", round(100 * train_frac))
round(100 * res_size[, acc_cols])

### Summary plot

Test accuracy of the full and partial models for each split. Each panel is one model; the horizontal axis lists the splits. The dashed black line is the full model. In the middle panel, grey circles mark the Firth partial model with (A,S) dropped on the paper's split, which appears in both groups.

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 6.5)
res_all <- rbind(res_pos, res_size)
col_m <- c("drop (A,M)" = "#E69F00", "drop (A,S)" = "#0072B2", "drop (M,S)" = "#009E73", full = "black")
lty_m <- c(1, 1, 1, 2); lwd_m <- c(1.5, 1.5, 1.5, 2.5)   # full model drawn last, dashed
panel <- function(cols, main) {
  g1 <- seq_len(nrow(res_pos)); g2 <- nrow(res_pos) + seq_len(nrow(res_size))
  matplot(g1, 100 * res_all[g1, cols], type = "b", pch = 16, lty = lty_m, lwd = lwd_m, col = col_m, ylim = c(60, 100),
          xlim = c(1, nrow(res_all)), xaxt = "n", xlab = "", ylab = "test accuracy (%)", main = main)
  matlines(g2, 100 * res_all[g2, cols], type = "b", pch = 16, lty = lty_m, lwd = lwd_m, col = col_m)
  if (main == "Firth")                          # paper's split, (A,S) dropped
    points(nrow(res_pos) + c(0, 3), 100 * res_all[nrow(res_pos) + c(0, 3), cols[2]], cex = 2.2, col = "grey40")
  axis(1, at = seq_len(nrow(res_all)), labels = sub("test block at |train ", "", rownames(res_all)), cex.axis = 0.8)
  abline(v = nrow(res_pos) + 0.5, col = "grey60", lty = 3)
  mtext(c("position of test block", "training fraction"), side = 1, line = 2.5,
        at = c(nrow(res_pos) / 2 + 0.5, nrow(res_pos) + nrow(res_size) / 2 + 0.5), cex = 0.8)
}
par(mfrow = c(1, 3), mar = c(4.5, 4, 2.5, 0.5))
panel(c(paste("LDA", drop_lab), "LDA full"), "LDA")
panel(c(paste("Firth", drop_lab), "Firth full"), "Firth")
panel(c(paste("Firth", drop_lab, "+ T"), "Firth full"), "Firth, CV temperatures")
legend("bottomright", legend = names(col_m), col = col_m, pch = 16, lty = lty_m, lwd = lwd_m, bty = "n", cex = 0.85)

### Findings

* **LDA is stable.** In every split, no LDA partial model is more than 4 percentage points below the full LDA model.
* **Firth partial models are fragile, but which pair hurts depends on the split.** Of the 11 distinct splits (the paper's split appears in both groups), dropping a pair costs the Firth model more than 10 percentage points in 6, and the damaging pair varies. The paper's split, where dropping (A,S) gives 67% (a loss of 28 points), is the most extreme case.
* **Temperature scaling mostly recovers the accuracy.** With temperatures chosen by blocked cross-validation on the training clips, the Firth partial models are within 4 points of the full model in every split except one (test block at 50%, (A,M) dropped: a loss of 15 points).

In [ ]:
## Largest loss of a partial model relative to the full model of the same type, per split.
loss <- function(m, suffix = "") sapply(seq_len(nrow(res_all)), function(r)
  max(res_all[r, paste0(m, " full")] - res_all[r, paste0(m, " ", drop_lab, suffix)]))
round(100 * cbind(LDA = loss("LDA"), Firth = loss("Firth"), "Firth + T" = loss("Firth", " + T")), 0) |>
  `rownames<-`(rownames(res_all))